# Linear Regression 

In [1]:

# 1) Import
import pandas as pd
import numpy as np
import time
import psutil
import os

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [2]:

# 2) Load train & test 
train_df = pd.read_csv("../data/processed/train_data_final.csv")
test_df  = pd.read_csv("../data/processed/test_data_final.csv")

In [3]:

# 3) Split features/target in train, test
TARGET = "quantity_sold"
assert TARGET in train_df.columns, f"Không tìm thấy cột target trong train: {TARGET}"

# train set
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

# test set
has_y_test = TARGET in test_df.columns
X_test = test_df.drop(columns=[TARGET]) if has_y_test else test_df.copy()
y_test = test_df[TARGET] if has_y_test else None

# target column & feature columns
print("Target column:", TARGET)
print("Number of features:", X_train.shape[1])
print("Feature columns:", list(X_train.columns))

Target column: quantity_sold
Number of features: 54
Feature columns: ['price', 'original_price', 'discount_rate', 'rating_average', 'review_count', 'is_return_policy', 'is_freeship_xtra', 'is_authentic', 'image_count', 'video_count', 'is_brand', 'store_review_count', 'total_follower', 'is_official', 'reputation_score', 'trust_level', 'review_to_sold_ratio', 'shop_potential', 'total_visuals', 'has_video', 'discount_amount', 'price_vs_category', 'hot_keyword_count', 'name_length', 'name_word_count', 'category_root_name_Bách Hóa Online', 'category_root_name_Chăm sóc nhà cửa', 'category_root_name_Giày - Dép nam', 'category_root_name_Giày - Dép nữ', 'category_root_name_Laptop – Máy Vi Tính – Linh kiện', 'category_root_name_Làm Đẹp - Sức Khỏe', 'category_root_name_Máy Ảnh - Máy Quay Phim', 'category_root_name_NGON', 'category_root_name_Nhà Cửa - Đời Sống', 'category_root_name_Nhà Sách Tiki', 'category_root_name_Phụ kiện thời trang', 'category_root_name_Thiết bị số - Phụ kiện số', 'category_r

In [4]:

# 4) Load + train model
model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
])

# time & space used for training
process = psutil.Process(os.getpid())

mem_before = process.memory_info().rss / (1024 ** 2)  # measured in MB
start_time = time.time()

model.fit(X_train, y_train) # model

end_time = time.time()
mem_after = process.memory_info().rss / (1024 ** 2)   # measured in MB

print("Training done.")
print(f"Training time: {end_time - start_time:.4f} seconds")
print(f"Memory used: {mem_after - mem_before:.2f} MB")


Training done.
Training time: 0.0600 seconds
Memory used: 1.67 MB


In [5]:

# 5) Predict on test
y_pred = model.predict(X_test)
print("Pred shape:", y_pred.shape)

# Evaluation metrics: MAE, MSE, RMSE, R2
if y_test is not None:
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    r2 = r2_score(y_test, y_pred)
    print(f"MAE : {mae:.4f}")
    print(f"MSE : {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2  : {r2:.4f}")


Pred shape: (3963,)
MAE : 0.5245
MSE : 0.5254
RMSE: 0.7249
R2  : 0.8996


In [6]:

# 6) Export predictions
# Print y_true & y_pred
out = pd.DataFrame(
    {
        "quantity_sold_ground_truth": y_test.values,
        "quantity_sold_predicted": y_pred
    },
    index=test_df.index   
)

out.to_csv("lr_predictions.csv", index=True)
out.head()



,quantity_sold_ground_truth,quantity_sold_predicted
0,3.218876,2.893549
1,0.000000,0.447549
2,1.791759,1.848326
3,3.465736,3.411308
4,0.000000,0.367485


In [7]:
# 7) Percentage of correct predictions (regression-style accuracy)

# tolerance = 10% (it can be: 0.05, 0.2, ...)
tolerance = 0.10

# Tránh chia cho 0
mask = y_test != 0
y_true_nonzero = y_test[mask]
y_pred_nonzero = y_pred[mask]

relative_error = np.abs(y_pred_nonzero - y_true_nonzero) / y_true_nonzero

accuracy_percent = (relative_error <= tolerance).mean() * 100

print(f"Prediction accuracy within ±{int(tolerance*100)}%: {accuracy_percent:.2f}%")


Prediction accuracy within ±10%: 38.90%
